%md
# 04 · Silver patterns
Parses the IBM pattern file into one row per laundering transaction, with its pattern type and attempt ID, linked to silver txn_id. Used for evaluation only — never exposed to the agent.

In [0]:
from pyspark.sql import functions as F

BRONZE = "workspace.aml_bronze"
SILVER = "workspace.aml_silver"

lines = spark.table(f"{BRONZE}.pattern_lines").orderBy("line_id")

In [0]:
typed_lines = lines.withColumn(
    "line_type",
    F.when(F.col("raw_line").startswith("BEGIN"), "begin")
     .when(F.col("raw_line").startswith("END"), "end")
     .when(F.trim("raw_line") == "", "blank")
     .otherwise("transaction"),
)

display(typed_lines.groupBy("line_type").count().orderBy("line_type"))

In [0]:
display(
    typed_lines
    .filter("line_type = 'begin'")
    .groupBy("raw_line")
    .count()
    .orderBy(F.desc("count"))
)

In [0]:
from pyspark.sql import Window

order_window = Window.orderBy("line_id").rowsBetween(Window.unboundedPreceding, Window.currentRow)

with_attempt = typed_lines.withColumn(
    "attempt_id",
    F.sum(F.when(F.col("line_type") == "begin", 1).otherwise(0)).over(order_window),
)

In [0]:
attempts = (
    with_attempt
    .filter("line_type = 'begin'")
    .select(
        "attempt_id",
        F.regexp_extract("raw_line", r"BEGIN LAUNDERING ATTEMPT - ([A-Z-]+)", 1).alias("pattern_type"),
        F.trim(F.regexp_extract("raw_line", r":\s*(.*)$", 1)).alias("pattern_detail"),
    )
)

display(attempts.groupBy("pattern_type").count().orderBy("pattern_type"))

In [0]:
pattern_txns = (
    with_attempt
    .filter("line_type = 'transaction'")
    .join(attempts, on="attempt_id", how="left")
)

display(pattern_txns.groupBy("pattern_type").count().orderBy("pattern_type"))

In [0]:
raw_cols = [
    "timestamp", "from_bank", "from_account", "to_bank", "to_account",
    "amount_received", "receiving_currency", "amount_paid",
    "payment_currency", "payment_format", "is_laundering",
]

parts = F.split(F.col("raw_line"), ",")

pattern_raw = pattern_txns.select(
    "line_id",
    "attempt_id",
    "pattern_type",
    "pattern_detail",
    *[F.trim(parts.getItem(i)).alias(c) for i, c in enumerate(raw_cols)],
)

display(pattern_raw.limit(5))

In [0]:
def normalize_bank(col):
    # Canonical bank ID: remove leading zeros, keep a single "0" if all zeros
    return F.regexp_replace(col, r"^0+(?=.)", "")

pattern_typed = pattern_raw.select(
    "line_id",
    "attempt_id",
    "pattern_type",
    "pattern_detail",
    F.expr("try_to_timestamp(timestamp, 'yyyy/MM/dd HH:mm')").alias("txn_timestamp"),
    normalize_bank(F.col("from_bank")).alias("from_bank"),
    F.col("from_account"),
    normalize_bank(F.col("to_bank")).alias("to_bank"),
    F.col("to_account"),
    F.expr("try_cast(amount_received AS DECIMAL(24,8))").alias("amount_received"),
    F.col("receiving_currency"),
    F.expr("try_cast(amount_paid AS DECIMAL(24,8))").alias("amount_paid"),
    F.col("payment_currency"),
    F.col("payment_format"),
)

fact_cols = [
    "txn_timestamp", "from_bank", "from_account", "to_bank", "to_account",
    "amount_received", "receiving_currency", "amount_paid",
    "payment_currency", "payment_format",
]

pattern_hashed = pattern_typed.withColumn(
    "txn_hash",
    F.substring(
        F.sha2(F.concat_ws("|", *[F.col(c).cast("string") for c in fact_cols]), 256),
        1,
        16,
    ),
)

In [0]:
silver_txns = spark.table(f"{SILVER}.transactions").select("txn_id", "txn_hash", "is_laundering")

matched = pattern_hashed.join(silver_txns, on="txn_hash", how="left")

display(
    matched.agg(
        F.count("*").alias("rows_after_join"),
        F.countDistinct("line_id").alias("pattern_lines"),
        F.sum(F.col("txn_id").isNull().cast("int")).alias("unmatched"),
        F.sum((F.col("is_laundering") == 1).cast("int")).alias("silver_says_laundering"),
    )
)

In [0]:
pattern_labels = matched.select(
    "txn_id",
    "attempt_id",
    "pattern_type",
    "pattern_detail",
    "txn_timestamp",
    "from_bank",
    "from_account",
    "to_bank",
    "to_account",
    "amount_paid",
    "payment_currency",
    "line_id",
    F.current_timestamp().alias("_processed_at"),
)

display(pattern_labels.limit(5))

In [0]:
total_rows = pattern_labels.count()
unique_txns = pattern_labels.select("txn_id").distinct().count()
total_attempts = pattern_labels.select("attempt_id").distinct().count()
pattern_types = pattern_labels.select("pattern_type").distinct().count()
null_ids = pattern_labels.filter(F.col("txn_id").isNull()).count()

print(f"Rows:           {total_rows:,}")
print(f"Unique txn_ids: {unique_txns:,}")
print(f"Attempts:       {total_attempts:,}")
print(f"Pattern types:  {pattern_types}")
print(f"Missing IDs:    {null_ids}")

assert total_rows == 3209, "Expected 3,209 pattern transactions"
assert null_ids == 0, "Some pattern transactions did not link to silver"
assert total_attempts == 370, "Expected 370 laundering attempts"
assert pattern_types == 8, "Expected 8 pattern types"

print("Quality gate passed")

spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.aml_eval")

(
    pattern_labels.write
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable("workspace.aml_eval.pattern_labels")
)